In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from tqdm import tqdm
import matplotlib.pyplot as plt


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


In [3]:
CSV_PATH = "/home/kishore/nafld_kcnn_project/data/clinical/roboflow_matched_real.csv"

df = pd.read_csv(CSV_PATH)

REQUIRED = [
    "slide_id", "patient_id_real",
    "steatosis_grade", "inflammation_grade",
    "ballooning_grade", "fibrosis_stage",
    "nas_score", "age", "bmi",
    "followup_years",
    "progression_event", "progression_risk"
]

df = df[REQUIRED].dropna()
print("Samples:", len(df))
print("Patients:", df["patient_id_real"].nunique())


Samples: 6000
Patients: 2000


In [4]:
groups = df["patient_id_real"].astype(str)

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

gss2 = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, val_idx = next(
    gss2.split(train_df, groups=train_df["patient_id_real"].astype(str))
)

val_df   = train_df.iloc[val_idx]
train_df = train_df.iloc[train_idx]

print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))


Train / Val / Test: 3840 960 1200


In [5]:
class ProgressionDataset(Dataset):
    def __init__(self, df):
        # Histology block
        self.histology = torch.tensor(
            df[[
                "steatosis_grade",
                "inflammation_grade",
                "ballooning_grade",
                "nas_score"
            ]].values,
            dtype=torch.float32
        )

        # Fibrosis block
        self.fibrosis = torch.tensor(
            df[["fibrosis_stage"]].values,
            dtype=torch.float32
        )

        # Clinical block
        self.clinical = torch.tensor(
            df[["age", "bmi", "followup_years"]].values,
            dtype=torch.float32
        )

        self.y_event = torch.tensor(
            df["progression_event"].values,
            dtype=torch.float32
        )

        self.y_risk = torch.tensor(
            df["progression_risk"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.y_event)

    def __getitem__(self, idx):
        return (
            self.histology[idx],
            self.fibrosis[idx],
            self.clinical[idx],
            self.y_event[idx],
            self.y_risk[idx],
        )


In [6]:
train_loader = DataLoader(ProgressionDataset(train_df), batch_size=64, shuffle=True)
val_loader   = DataLoader(ProgressionDataset(val_df), batch_size=64)
test_loader  = DataLoader(ProgressionDataset(test_df), batch_size=64)


In [7]:
class StructuredRiskNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.histology_enc = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU()
        )

        self.fibrosis_enc = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU()
        )

        self.clinical_enc = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(32 + 16 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Heads
        self.event_head = nn.Linear(64, 1)
        self.risk_head  = nn.Linear(64, 1)
        self.uncert_head = nn.Linear(64, 1)  # uncertainty

    def forward(self, h, f, c):
        h = self.histology_enc(h)
        f = self.fibrosis_enc(f)
        c = self.clinical_enc(c)

        z = self.fusion(torch.cat([h, f, c], dim=1))

        event = torch.sigmoid(self.event_head(z))
        risk  = torch.sigmoid(self.risk_head(z))
        uncert = F.softplus(self.uncert_head(z))  # positive uncertainty

        return event.squeeze(), risk.squeeze(), uncert.squeeze()


In [8]:
model = StructuredRiskNet().to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

bce = nn.BCELoss()
mse = nn.MSELoss()


In [9]:
def train_epoch(loader):
    model.train()
    total_loss = 0.0

    for h, f, c, y_e, y_r in loader:
        h, f, c = h.to(DEVICE), f.to(DEVICE), c.to(DEVICE)
        y_e, y_r = y_e.to(DEVICE), y_r.to(DEVICE)

        p_e, p_r, u = model(h, f, c)

        # ---- Classification loss ----
        loss_event = bce(p_e, y_e)

        # ---- Aleatoric regression loss (CORRECT) ----
        # u is positive uncertainty (sigma)
        loss_risk = ((p_r - y_r) ** 2) / (2 * u**2 + 1e-6) + torch.log(u + 1e-6)
        loss_risk = loss_risk.mean()

        # ---- Total scalar loss ----
        loss = loss_event + loss_risk

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [10]:
def evaluate(loader):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for h, f, c, y_e, _ in loader:
            p, _, _ = model(
                h.to(DEVICE),
                f.to(DEVICE),
                c.to(DEVICE)
            )
            y_true.extend(y_e.numpy())
            y_prob.extend(p.cpu().numpy())

    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, np.array(y_prob) > 0.5)
    brier = brier_score_loss(y_true, y_prob)

    return auc, acc, brier


In [11]:
best_auc = 0

for epoch in range(1, 101):
    loss = train_epoch(train_loader)
    auc, acc, brier = evaluate(val_loader)

    print(
        f"Epoch {epoch:03d} | "
        f"loss={loss:.3f} | "
        f"val_auc={auc:.3f} | "
        f"val_acc={acc:.3f} | "
        f"brier={brier:.3f}"
    )

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "models/progression/structured_risk_net.pth")


Epoch 001 | loss=0.131 | val_auc=0.549 | val_acc=0.791 | brier=0.166
Epoch 002 | loss=-0.897 | val_auc=0.604 | val_acc=0.791 | brier=0.163
Epoch 003 | loss=-1.144 | val_auc=0.660 | val_acc=0.791 | brier=0.160
Epoch 004 | loss=-1.424 | val_auc=0.703 | val_acc=0.791 | brier=0.156
Epoch 005 | loss=-1.589 | val_auc=0.711 | val_acc=0.791 | brier=0.154
Epoch 006 | loss=-1.709 | val_auc=0.711 | val_acc=0.791 | brier=0.155
Epoch 007 | loss=-1.761 | val_auc=0.708 | val_acc=0.791 | brier=0.155
Epoch 008 | loss=-1.818 | val_auc=0.731 | val_acc=0.791 | brier=0.151
Epoch 009 | loss=-1.882 | val_auc=0.737 | val_acc=0.791 | brier=0.151
Epoch 010 | loss=-2.003 | val_auc=0.740 | val_acc=0.791 | brier=0.150
Epoch 011 | loss=-2.036 | val_auc=0.737 | val_acc=0.791 | brier=0.149
Epoch 012 | loss=-2.067 | val_auc=0.743 | val_acc=0.791 | brier=0.148
Epoch 013 | loss=-2.142 | val_auc=0.738 | val_acc=0.791 | brier=0.149
Epoch 014 | loss=-2.189 | val_auc=0.744 | val_acc=0.791 | brier=0.149
Epoch 015 | loss=-2.1

In [12]:
model.load_state_dict(
    torch.load("models/progression/structured_risk_net.pth")
)

auc, acc, brier = evaluate(test_loader)

print("\nTEST RESULTS")
print("AUC:", auc)
print("Accuracy:", acc)
print("Brier:", brier)



TEST RESULTS
AUC: 0.6629163713678242
Accuracy: 0.83
Brier: 0.13466483528812645
